In [55]:
import os 
import json
import csv
import subprocess
import psycopg2
from datetime import datetime

In [56]:
EXECUTION_POLICY = {
    "allowed_step_types": [
        "database_read",
        "script_execution",
        "notification"
    ],
    "sandbox_dir": "sandbox/runtime"
}

SQL_POLICY = {
    "allowed_commands": ["SELECT", "WITH"],
    "require_limit": True,
    "max_rows": 1000
}

SCRIPT_POLICY = {
    "allowed_language": "python",
    "timeout_seconds": 30,
    "allowed_paths": ["sandbox/runtime", "scripts"]
}

ALLOWED_EMAILS = ["team@example.com"]

In [57]:
execution_state = {
    "files_created" : [],
    "logs" : [],
    "status" : "INIT"
} 

In [58]:
def validation_workflow(workflow):
    if "steps" not in workflow:
        raise ValueError("Workflow Missing Steps")
    
    

    for step in workflow["steps"]:
        step_type = step.get("type")

        if step_type not in EXECUTION_POLICY["allowed_step_types"]:
            raise ValueError("steps not alowed")
        
        else:
            print("Executed Successfully")
            

In [59]:
def sql_policy(query):
    query_upper = query.upper()

    if not any(query_upper.startswith(cmd) for cmd in SQL_POLICY["allowed_commands"]):
        raise ValueError("Query Not Allowed")
    
    if SQL_POLICY["require_limit"] and "LIMIT" not in query_upper:
        raise ValueError("Query should be in limit")




In [60]:
from dotenv import load_dotenv

load_dotenv()

True

In [61]:
NEON_URL = os.getenv("Neon_URL")

In [62]:
def db_read(step):
    conn_str = NEON_URL
    if not conn_str:
        raise RuntimeError("NEON_READONLY_URL not set")

    query = step["query"]
    sql_policy(query)

    output_file = os.path.join(EXECUTION_POLICY["sandbox_dir"], step["output"])

    conn = psycopg2.connect(conn_str)
    cursor = conn.cursor()
    cursor.execute(query)

    rows = cursor.fetchmany(SQL_POLICY["max_rows"])
    headers = [desc[0] for desc in cursor.description]

    with open(output_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        writer.writerows(rows)

    cursor.close()
    conn.close()

    execution_state["files_created"].append(output_file)
    print(f"DB data written to {output_file}")

In [63]:
def run_script(step):
    script_path = os.path.join("scripts", step["script"])

    output_files = step["outputs"]

    cmd = ["python", script_path] + step["inputs"] + output_files

    try:
        subprocess.run(cmd, 
                       check = True,
                       timeout = SCRIPT_POLICY["timeout_seconds"])
        
    except subprocess.TimeoutExpired:
        raise RuntimeError("Script Execution TImeout")
    
    for f in output_files:
        execution_state["files_created"].append(os.path.join(EXECUTION_POLICY["sandbox_dir"],f))

    print("Script Execution Successfully")

    

    

In [64]:
def rollback():
    print("Rolling Back Execution")

    for f in execution_state["files_created"]:
        try:
            os.remove(f)
            print("File Deleted")

        except FileNotFoundError:
            pass


In [65]:
def audit():
    execution_state["status"] = "COMPLETED"

    execution_state["logs"].append(f"EXECUTION COMPLETED AT {datetime.utcnow()}")

    print("Audit Complete") 


In [66]:
def workflow_execution(workflow):
    os.makedirs(EXECUTION_POLICY["sandbox_dir"], exist_ok=True)
    execution_state["status"] = "RUNNING"

    try:
        validation_workflow(workflow)

        for step in workflow["steps"]:
            if step["type"] == "database_read":
                db_read(step)

            elif step["type"] == "script_execution":
                run_script(step)

        audit()
                

    except Exception as e:
        execution_state['status'] = "Failed"
        rollback()

        print("WorkFlow Execution Failed")

        raise

In [67]:
workflow = {
    "workflow_id": "report-001",
    "steps": [
        {
            "type": "database_read",
            "query": "SELECT date, revenue FROM sales LIMIT 100",
            "output": "sales_raw.csv"
        },
        {
            "type": "script_execution",
            "script": "generate_report.py",
            "inputs": ["sandbox/runtime/sales_raw.csv"],
            "outputs": ["sandbox/runtime/sales_report.csv"]
        },
        {
            "type": "notification",
            "to": ["team@example.com"],
            "attachment": "sandbox/runtime/sales_report.csv"
        }
    ]
}

In [68]:
workflow_execution(workflow)

Executed Successfully
Executed Successfully
Executed Successfully
DB data written to sandbox/runtime\sales_raw.csv
Script Execution Successfully
Audit Complete


C:\Users\parth.kadoo\AppData\Local\Temp\ipykernel_14192\91069291.py:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  execution_state["logs"].append(f"EXECUTION COMPLETED AT {datetime.utcnow()}")
